In [ ]:
# Proximity Funktion mit Gauss-Peaks
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import numpy as np
from scipy.integrate import simpson
from scipy.stats import gaussian_kde

def proximity_max_freq_gaussian(
    x,
    sharpness_base=10.0,
    sharpness_growth=1.0,
    decay=0.1,
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    neighbor_peaks=2,   # wie viele Peaks links+rechts von x getestet werden
    norm_x=1.0,
    output_max=1.0
):
    """
    Optimierte Gaussian-Proximity-Funktion:
    - prüft nur die Ratio-Peaks, die in der Nähe von x liegen (k/f ± neighbor_peaks)
    - x: Skalar oder Array
    - Rückgabe: Array gleicher Form
    """
    x = np.array(x, dtype=float)
    all_freq_values = []

    for f in range(1, max_freq + 1):
        weight = 1.0 / (f ** weight_exponent)
        gauss_max = np.zeros_like(x, dtype=float)

        # nächstliegender Peak-Index zu jedem x
        k_center = np.round(x * f).astype(int)

        for offset in range(-neighbor_peaks, neighbor_peaks + 1):
            k = k_center + offset
            valid = k > 0  # negative oder 0 Peaks sind Unsinn

            if not np.any(valid):
                continue

            mu = k[valid] / f
            sharpness = sharpness_base * (f ** freq_sharpness_exp) * (k[valid] ** sharpness_growth)
            sigma = 1.0 / (sharpness + 1e-9)
            mu_decay = 1.0 / (mu ** decay)

            # Broadcasting über alle validen Peaks
            g = np.exp(-0.5 * ((x[..., None] - mu[None, ...]) / sigma[None, ...]) ** 2)
            contrib = weight * mu_decay * g

            # Max über Peaks in Nachbarschaft
            gauss_max = np.maximum(gauss_max, np.max(contrib, axis=-1))

        all_freq_values.append(gauss_max)

    result = np.maximum.reduce(all_freq_values)

    # Normierung
    max_at_norm_x = np.max([
        (1.0 / (f ** weight_exponent))
        for f in range(1, max_freq + 1)
    ])
    return result / (max_at_norm_x + 1e-9) * output_max

def plot_proximity_max_freq_gaussian(
    sharpness_base=10.0,
    sharpness_growth=1.0,
    decay=0.5,
    max_freq=6,
    neighbor_peaks=2,
    weight_exponent=1.0,
    freq_sharpness_exp=0.5
):
    x_vals = np.linspace(1, 5, 2000)
    y_vals = proximity_max_freq_gaussian(
        x_vals,
        sharpness_base=sharpness_base,
        sharpness_growth=sharpness_growth,
        decay=decay,
        max_freq=max_freq,
        weight_exponent=weight_exponent,
        freq_sharpness_exp=freq_sharpness_exp,
        neighbor_peaks=2,
        output_max=1.0,
    )

    mean_prox = simpson(y_vals, x=x_vals) / (x_vals[-1] - x_vals[0])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Proximity-Funktion mit k- und f-abhängigen Gauß-Peaks", fontsize=16, y=1.02)

    # Funktion plotten
    axes[0].plot(x_vals, y_vals, label=f"Proximity (∫mean ≈ {mean_prox:.3f})")
    axes[0].axhline(mean_prox, color='red', linestyle='--', linewidth=2,
                    label=f"∫mean ≈ {mean_prox:.3f}")
    axes[0].set_xlabel("x (Verhältnis)")
    axes[0].set_ylabel("Proximity")
    axes[0].set_title("Proximity mit k-Decay & f-Weight")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, linestyle="--", alpha=0.3)
    axes[0].legend()

    # Histogramm + KDE
    counts, _, _ = axes[1].hist(
        y_vals, bins=50, range=(0, 1),
        alpha=0.4, color="skyblue", density=True, label="Histogramm"
    )
    kde = gaussian_kde(y_vals)
    x_dens = np.linspace(0, 1, 5000)
    kde_vals = kde(x_dens)

    scaling_factor = max(counts) / max(kde_vals)
    axes[1].plot(x_dens, kde_vals * scaling_factor,
                 color="darkblue", lw=2, label="Dichte (KDE)")
    axes[1].axvline(mean_prox, color='red', linestyle='--',
                    linewidth=2, label=f"∫mean ≈ {mean_prox:.3f}")
    axes[1].set_title("Proximity-Verteilung")
    axes[1].set_xlabel("Proximity")
    axes[1].set_ylabel("Normierte Häufigkeit")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()

# Interaktive Kontrolle
interact(
    plot_proximity_max_freq_gaussian,
    sharpness_base=FloatSlider(min=1, max=50, step=1, value=10, description="Sharpness Base"),
    sharpness_growth=FloatSlider(min=0, max=3, step=0.1, value=1, description="Sharpness Growth (k)"),
    decay=FloatSlider(min=0, max=3, step=0.05, value=0.5, description="Decay (mu)"),
    max_freq=IntSlider(min=1, max=6, step=1, value=4, description="Max Frequency"),
    weight_exponent=FloatSlider(min=0, max=3, step=0.1, value=1, description="Weight Exponent (f)"),
    freq_sharpness_exp=FloatSlider(min=0, max=3, step=0.1, value=1, description="Freq Sharpness Exp (f)"),
)

interactive(children=(FloatSlider(value=10.0, description='Sharpness Base', max=50.0, min=1.0, step=1.0), Floa…

<function __main__.plot_proximity_max_freq_gaussian(sharpness_base=10.0, sharpness_growth=1.0, decay=0.5, max_freq=6, neighbor_peaks=2, weight_exponent=1.0, freq_sharpness_exp=0.5)>

In [3]:
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import numpy as np
from scipy.integrate import simpson
from scipy.stats import gaussian_kde

# -------- Optimierte Gaussian-Proximity mit echter Neighbor-Logik --------
def proximity_max_freq_gaussian_fast(
    x,
    sharpness_base=10.0,
    sharpness_growth=1.0,
    decay=0.1,
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    neighbor_peaks=2,
    x_min=0.5,
    x_max=5.0,
    output_max=1.0
):
    """
    Gaussian-based proximity function with true neighbor-peak optimization.
    - Precomputes all relevant (k/f) peaks once
    - For each x, only evaluates the closest `neighbor_peaks`
    Complexity: O(N * neighbor_peaks)
    """
    x = np.array(x, dtype=float)

    # --- 1) Precompute all possible peaks (mu = k/f) in range ---
    peak_list = []
    for f in range(1, max_freq + 1):
        k_max = int(np.ceil(x_max * f))
        for k in range(1, k_max + 1):
            mu = k / f
            if mu < x_min or mu > x_max:
                continue
            weight = 1.0 / (f ** weight_exponent)
            sharpness = sharpness_base * (f ** freq_sharpness_exp) * (k ** sharpness_growth)
            sigma = 1.0 / (sharpness + 1e-9)
            mu_decay = 1.0 / (mu ** decay)
            peak_list.append((mu, sigma, weight * mu_decay))

    peak_list = np.array(peak_list, dtype=float)  # (N, 3): (mu, sigma, amplitude)
    mus, sigmas, amps = peak_list[:, 0], peak_list[:, 1], peak_list[:, 2]

    # --- 2) For each x, evaluate only closest peaks ---
    result = np.zeros_like(x, dtype=float)
    for i, xi in enumerate(x):
        # Find nearest peaks
        idx = np.argpartition(np.abs(mus - xi), neighbor_peaks)[:neighbor_peaks]
        g = np.exp(-0.5 * ((xi - mus[idx]) / sigmas[idx]) ** 2)
        result[i] = np.max(amps[idx] * g)

    # --- 3) Normalize ---
    result /= np.max(result) + 1e-9
    return result * output_max


# -------- Plotting wrapper --------
def plot_proximity_max_freq_gaussian(
    sharpness_base=10.0,
    sharpness_growth=1.0,
    decay=0.5,
    max_freq=6,
    neighbor_peaks=2,
    weight_exponent=1.0,
    freq_sharpness_exp=0.5
):
    x_vals = np.linspace(1, 5, 2000)
    y_vals = proximity_max_freq_gaussian_fast(
        x_vals,
        sharpness_base=sharpness_base,
        sharpness_growth=sharpness_growth,
        decay=decay,
        max_freq=max_freq,
        weight_exponent=weight_exponent,
        freq_sharpness_exp=freq_sharpness_exp,
        neighbor_peaks=neighbor_peaks,
        x_min=1.0,
        x_max=5.0,
        output_max=1.0,
    )

    mean_prox = simpson(y_vals, x=x_vals) / (x_vals[-1] - x_vals[0])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Optimized Gaussian Proximity Function", fontsize=16, y=1.02)

    # Funktion plotten
    axes[0].plot(x_vals, y_vals, label=f"Proximity (∫mean ≈ {mean_prox:.3f})")
    axes[0].axhline(mean_prox, color='red', linestyle='--', linewidth=2,
                    label=f"∫mean ≈ {mean_prox:.3f}")
    axes[0].set_xlabel("x (ratio)")
    axes[0].set_ylabel("Proximity")
    axes[0].set_title("Proximity with k-decay & f-weight")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, linestyle="--", alpha=0.3)
    axes[0].legend()

    # Histogramm + KDE
    counts, _, _ = axes[1].hist(
        y_vals, bins=50, range=(0, 1),
        alpha=0.4, color="skyblue", density=True, label="Histogram"
    )
    kde = gaussian_kde(y_vals)
    x_dens = np.linspace(0, 1, 5000)
    kde_vals = kde(x_dens)

    scaling_factor = max(counts) / max(kde_vals)
    axes[1].plot(x_dens, kde_vals * scaling_factor,
                 color="darkblue", lw=2, label="Density (KDE)")
    axes[1].axvline(mean_prox, color='red', linestyle='--',
                    linewidth=2, label=f"∫mean ≈ {mean_prox:.3f}")
    axes[1].set_title("Proximity distribution")
    axes[1].set_xlabel("Proximity")
    axes[1].set_ylabel("Normalized frequency")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()


# -------- Interaktive Kontrolle --------
interact(
    plot_proximity_max_freq_gaussian,
    sharpness_base=FloatSlider(min=1, max=50, step=1, value=10, description="Sharpness Base"),
    sharpness_growth=FloatSlider(min=0, max=3, step=0.1, value=1, description="Sharpness Growth (k)"),
    decay=FloatSlider(min=0, max=3, step=0.05, value=0.5, description="Decay (mu)"),
    max_freq=IntSlider(min=1, max=10, step=1, value=6, description="Max Frequency"),
    neighbor_peaks=IntSlider(min=1, max=10, step=1, value=2, description="Neighbor Peaks"),
    weight_exponent=FloatSlider(min=0, max=3, step=0.1, value=1, description="Weight Exponent (f)"),
    freq_sharpness_exp=FloatSlider(min=0, max=3, step=0.1, value=0.5, description="Freq Sharpness Exp (f)"),
)


interactive(children=(FloatSlider(value=10.0, description='Sharpness Base', max=50.0, min=1.0, step=1.0), Floa…

<function __main__.plot_proximity_max_freq_gaussian(sharpness_base=10.0, sharpness_growth=1.0, decay=0.5, max_freq=6, neighbor_peaks=2, weight_exponent=1.0, freq_sharpness_exp=0.5)>